# 1. import library

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMBA_NUM_THREADS"] = "1"

import joblib
from joblib import Parallel, delayed
import random

import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import gaussian_kde, truncnorm
from numba import njit, prange, float64

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from collections import defaultdict
from svgutils.compose import Figure, SVG, Text

# 2. import data

## 2.1. rawdata

In [ ]:
# folder path
prefix = "../"
name_1 = "0_batch_experiment_data"
name_2_1 = "summary_CFS.csv"
name_2_2 = "summary_noCFS.csv"
file_name = os.path.join(prefix, name_1, name_2_1)
file_name_2 = os.path.join(prefix, name_1, name_2_2)

exp_sup = pd.read_csv(file_name)
exp_noSup = pd.read_csv(file_name_2)

## 2.2. mutate

In [ ]:
# CFS+ data
exp_sup_mutate = exp_sup.copy()
exp_sup_mutate = exp_sup_mutate.query('Specie == "PY1" and Condition == "supernatant 10%"')
exp_sup_mutate['N'] = exp_sup_mutate['N'].astype(str)
exp_sup_mutate['ID'] = exp_sup_mutate['ID'].astype(str)

# initial nitrite concentration
initial_concentrations = {
    "N1": 0.070957882,
    "N2": 0.039015614,
    "N3": 0.063268077
}

def compute_nitrite_production(row):
    if row["Condition"] == "supernatant 10%":
        if row["N"] == "1":
            val = row["Nitrite"] - initial_concentrations["N1"]
        elif row["N"] == "2":
            val = row["Nitrite"] - initial_concentrations["N2"]
        elif row["N"] == "3":
            val = row["Nitrite"] - initial_concentrations["N3"]
        else:
            return row["Nitrite"]
        return val if val > 0 else 0
    else:
        return row["Nitrite"]

exp_sup_mutate["Nitrite_production"] = exp_sup_mutate.apply(compute_nitrite_production, axis=1)
exp_sup_mutate["init_cell_num"] = exp_sup_mutate["CellDensity"].str.replace("^", "**", regex=False).map(lambda x: eval(x)) # cellDensity: cells / mL, culture volume 1mL
exp_sup_mutate['source'] = 'exp'

Nitrite_production_mM = exp_sup_mutate["Nitrite_production"]
Nitrite_production_pM = Nitrite_production_mM * 1e9 # 1e9: mM -> pM
Nitrite_production_pmol = Nitrite_production_pM * 1e-3 # total volume: 1e-3 L
exp_sup_mutate["produced_cell_num"] = Nitrite_production_pmol * 33.5 # yield: 33.5 cells/pmol

exp_sup_mutate["cell_num"] = exp_sup_mutate["init_cell_num"] + exp_sup_mutate["produced_cell_num"]

In [ ]:
# CFS- data
exp_noSup_mutate = exp_noSup.copy()
exp_noSup_mutate = exp_noSup_mutate.query('Specie == "PY1" and Condition == "Cat+"')
exp_noSup_mutate['N'] = exp_noSup_mutate['N'].astype(str)
exp_noSup_mutate['ID'] = exp_noSup_mutate['ID'].astype(str)

# initial nitrite concentration = 0

exp_noSup_mutate["Nitrite_production"] = exp_noSup_mutate["Nitrite"]
exp_noSup_mutate["init_cell_num"] = exp_noSup_mutate["CellDensity"].str.replace("^", "**", regex=False).map(lambda x: eval(x)) # cellDensity: cells / mL, culture volume 1mL
exp_noSup_mutate['source'] = 'exp'

Nitrite_production_mM = exp_noSup_mutate["Nitrite_production"]
Nitrite_production_pM = Nitrite_production_mM * 1e9 # 1e9: mM -> pM
Nitrite_production_pmol = Nitrite_production_pM * 1e-3 # total volume: 1e-3 L
exp_noSup_mutate["produced_cell_num"] = Nitrite_production_pmol * 33.5 # yield: 33.5 cells/pmol

exp_noSup_mutate["cell_num"] = exp_noSup_mutate["init_cell_num"] + exp_noSup_mutate["produced_cell_num"]

## 2.2. import single-cell params

### 2.2.1. import

In [ ]:
prefix_2 = "../../"
name = "3_regression/regression_result"
fit = pd.read_csv(os.path.join(prefix_2, name, '1_fit_results.csv'),
                  index_col='Model')
fit_3D = pd.read_csv(os.path.join(prefix_2, name, '2_fit_results_3D.csv'),
                     index_col='Model')
fit_DR = pd.read_csv(os.path.join(prefix_2, name, '3_fit_results_DR.csv'),
                     index_col='Field')
kde_data = np.load(os.path.join(prefix_2, name, "4_kde_training_data.npy"))
kde_bw = float(np.load(os.path.join(prefix_2, name, "5_kde_bandwidth.npy"))[0])
rf = joblib.load(os.path.join(prefix_2, name, "6_rf_model.pkl"))

In [ ]:
# reconstruct the KDE
kde_log = gaussian_kde(np.log(kde_data), bw_method=kde_bw)
  
# fitting parameters obtained from the experimental data(mean and std for each condition)
single_cell_exp_params = {
    # generation time, T
    'Gtime_mu_max': fit_3D.loc['generation_time_3D', 'p0'], 
    'Gtime_r1': fit_3D.loc['generation_time_3D', 'p1'], 'Gtime_r2': fit_3D.loc['generation_time_3D', 'p2'],
    'Gtime_mu_min': fit_3D.loc['generation_time_3D', 'p3'], 
    # sigma of generation time, T
    'Gtime_sigma_min': fit_3D.loc['generation_time_3D', 'sigma_p0'], 'Gtime_sigma_max': fit_3D.loc['generation_time_3D', 'sigma_p1'],
    'Gtime_sigma_x_c': fit_3D.loc['generation_time_3D', 'sigma_p2'], 'Gtime_sigma_k': fit_3D.loc['generation_time_3D', 'sigma_p3'],
    'Gtime_min': fit_3D.loc['generation_time_3D', 'z_min'],

    # elongation rate, α
    'alpha_mu_max': fit_3D.loc['elongation_rate_3D', 'p0'], 
    'alpha_mu_r1': fit_3D.loc['elongation_rate_3D', 'p1'], 'alpha_mu_r2': fit_3D.loc['elongation_rate_3D', 'p2'], 
    'alpha_mu_x01': fit_3D.loc['elongation_rate_3D', 'p3'], 'alpha_mu_x02': fit_3D.loc['elongation_rate_3D', 'p4'],
    # sigma of elongation rate, α
    'alpha_sigma_max': fit_3D.loc['elongation_rate_3D', 'sigma_p0'],
    'alpha_sigma_r': fit_3D.loc['elongation_rate_3D', 'sigma_p1'],
    'alpha_sigma_x0': fit_3D.loc['elongation_rate_3D', 'sigma_p2'],
    'alpha_max': fit_3D.loc['elongation_rate_3D', 'z_max'], 
    
    # max cell area, A_max
    # 'maxAd_max': fit.loc['max_Ad', 'max_val'], # use max cell area observed in the experiment in this simulation.
    'maxAd_mu': fit.loc['max_Ad', 'p0'], # when batch culture simulation, mean of max cell area was used.
    'Ad_sizer': fit.loc['Ad_sizer', 'p3'], 'Ad_sizer_sigma': fit.loc['Ad_sizer', 'sigma_p3'],
    
    # division ratio
    'divR_mu': fit_DR.loc['div_ratio', 'p1'], 'divR_sigma': fit_DR.loc['div_ratio', 'p2'], 
    'divR_min': fit_DR.loc['div_ratio', 'min_val'], 'divR_max': fit_DR.loc['div_ratio', 'max_val'],
    
    # KDE and RF model for initial cell properties
    'kde_log': kde_log, 'rf': rf
    } 


### 2.2.2. define parameter  

In [ ]:
# Volume convergence at cell birth（half of cell division）
convergence_cell_birth_volume = single_cell_exp_params["Ad_sizer"]/2 *0.75
# max cell volume
max_cell_volume = single_cell_exp_params['maxAd_mu'] * 0.75
print(convergence_cell_birth_volume)
print(max_cell_volume)

# Amount of nitrite produced per cell (pmol)
dK_per_cell = 1.0 / 33.5
# Ki(mM)
Ki = 1.0e-1

# initial ∆Vt CFS+ (µm^3/mL)
deltaVt0_CFS_1 = 0.070957882 *1e9 *1e-3 *33.5 *convergence_cell_birth_volume
deltaVt0_CFS_2 = 0.063950232 *1e9 *1e-3 *33.5 *convergence_cell_birth_volume
deltaVt0_CFS_3 = 0.063268077 *1e9 *1e-3 *33.5 *convergence_cell_birth_volume

# initial ∆Vt CFS- (µm^3/mL)
deltaVt0_noCFS_10_5 = convergence_cell_birth_volume * 1e5 * 0.2
deltaVt0_noCFS_10_3 = convergence_cell_birth_volume * 1e3 * 0.2
deltaVt0_noCFS_10_1 = convergence_cell_birth_volume * 1e1 * 0.2

# initial nitrite concentration (mM)
initial_nitrite_CFS_1 = 0.070957882
initial_nitrite_CFS_2 = 0.063950232
initial_nitrite_CFS_3 = 0.063268077

# nitrite detection limit (mM and pmol)
nitrite_detect_limit_mM = 1.0
nitrite_detect_limit_pmol = nitrite_detect_limit_mM * 1e9 * 1e-3 # total volume: 1e-3 L

# Number of repeated simulations
# (One full simulation run takes approximately 10 minutes on a Mac mini M4 with 32 GB memory.)
n_repeat = 100

# 3. functions

## 3.1. njit

In [ ]:
# calculate mean of T and α

mu_max_Gtime = float(single_cell_exp_params['Gtime_mu_max'])
r1_Gtime = float(single_cell_exp_params['Gtime_r1'])
r2_Gtime = float(single_cell_exp_params['Gtime_r2'])
mu_min_Gtime = float(single_cell_exp_params['Gtime_mu_min'])
@njit(parallel=True)
def compute_Gtime_3D_jit(biomass_production_density, cell_area_arr):
    out = np.empty(cell_area_arr.size, dtype=np.float64)
    for i in prange(cell_area_arr.size):
        out[i] = ((mu_max_Gtime - mu_min_Gtime)
                  * (biomass_production_density ** (-r1_Gtime)) 
                  * (cell_area_arr[i] ** (-r2_Gtime)) 
                  + mu_min_Gtime
                  )
    return out

mu_max_alpha = float(single_cell_exp_params['alpha_mu_max'])
r1_alpha = float(single_cell_exp_params['alpha_mu_r1'])
x01_alpha = float(single_cell_exp_params['alpha_mu_x01'])
r2_alpha = float(single_cell_exp_params['alpha_mu_r2'])
x02_alpha = float(single_cell_exp_params['alpha_mu_x02'])
@njit(parallel=True)
def compute_elongation_rate_3D_jit(biomass_production_density, cell_area_arr):
    elongation_rate = np.empty(cell_area_arr.size, dtype=np.float64)
    log_biomass_production_density = np.log10(biomass_production_density)
    for i in prange(cell_area_arr.size):
        z = (r1_alpha * (log_biomass_production_density - x01_alpha) 
             - r2_alpha * (cell_area_arr[i] - x02_alpha))
        elongation_rate[i] = mu_max_alpha / (1.0 + np.exp(-z))
    out = elongation_rate
    return out

In [ ]:
# calculate sigma of T and α

Gtime_sigma_min = float(single_cell_exp_params['Gtime_sigma_min'])
Gtime_sigma_max = float(single_cell_exp_params['Gtime_sigma_max'])
Gtime_sigma_x_c = float(single_cell_exp_params['Gtime_sigma_x_c'])
Gtime_sigma_k = float(single_cell_exp_params['Gtime_sigma_k'])
@njit(parallel=False)
def compute_Gtime_sigma_jit(biomass_production_density):
    Gtime_sigma = (
        Gtime_sigma_min
        +
        (Gtime_sigma_max - Gtime_sigma_min) 
        / (1.0 + 
           (biomass_production_density
            / Gtime_sigma_x_c) **Gtime_sigma_k
           )
        )
    
    return Gtime_sigma

alpha_sigma_max = float(single_cell_exp_params['alpha_sigma_max'])
alpha_sigma_r = float(single_cell_exp_params['alpha_sigma_r'])
alpha_sigma_x0 = float(single_cell_exp_params['alpha_sigma_x0'])
@njit(parallel=False)
def compute_elongation_rate_sigma_jit(biomass_production_density):
    log_biomass_production_density = np.log10(biomass_production_density)
    z = (alpha_sigma_r * (log_biomass_production_density - alpha_sigma_x0))
    alpha_sigma = alpha_sigma_max / (1.0 + np.exp(-z))
    return alpha_sigma

In [ ]:
# calculate elongation
@njit(parallel=True, fastmath=True)
def compute_elongation(cells_volume, cells_mu, dt_hour):
    n = len(cells_volume)
    volume_elongated = np.empty(n, dtype=np.float64)
    for i in prange(n):  # 並列ループ
        volume_elongated[i] = np.minimum(max_cell_volume,
                                         cells_volume[i] * np.exp(cells_mu[i] * dt_hour))
    return volume_elongated

In [ ]:
# calculate weibull hazard
@njit
def calculate_weibull_hazard_jit(age, timer_scale, shape):
    n = age.size
    h_t = np.zeros(n, dtype=np.float64)
    
    for i in range(n):
        t = age[i]
        if t > 0.0:
            t_scaled = t / timer_scale
            # f_t = (shape / timer_scale) * t_scaled**(shape - 1) * np.exp(-t_scaled**shape)
            # F_t = 1.0 - np.exp(-t_scaled**shape)
            # h_t[i] = f_t / (1.0 - F_t + 1e-12)  # ハザード関数
            h_t[i] = shape/timer_scale * t_scaled**(shape - 1)
        else:
            h_t[i] = 0.0
    return h_t

## 3.2. Common

In [ ]:
# calculate gTime, etc. 
rng = np.random.default_rng()

def r_truncnorm(mu, sigma, lower, upper, size):
    a = (lower - mu) / sigma
    b = (upper - mu) / sigma
    result = truncnorm.rvs(a, b, loc=mu, scale=sigma, size=size)

    return result

def compute_g_new_mu_new(biomass_production_density, volume_new_arr, n_div):
    area_new_arr = volume_new_arr/0.75
    
    g_mean = compute_Gtime_3D_jit(biomass_production_density, area_new_arr)
    g_sigma = compute_Gtime_sigma_jit(biomass_production_density)
    lower_bound = np.maximum(single_cell_exp_params["Gtime_min"],
                             g_mean - 3* g_sigma*1.0136)
    upper_bound = np.inf
    g_new = r_truncnorm(g_mean, g_sigma,
                        lower_bound, upper_bound, 
                        size=n_div).astype(np.float64)
    
    mu_mean = compute_elongation_rate_3D_jit(biomass_production_density, area_new_arr)
    mu_sigma = compute_elongation_rate_sigma_jit(biomass_production_density)
    lower_bound = np.maximum(0.0,
                             mu_mean - 3* mu_sigma*1.0136)
    upper_bound = np.minimum(single_cell_exp_params["alpha_max"],
                             mu_mean + 3* mu_sigma*1.0136)
    mu_new = r_truncnorm(mu_mean, mu_sigma,
                         lower_bound, upper_bound, 
                         size=n_div).astype(np.float64)
    
    return g_new, mu_new

def compute_sizer(n_div):
    lower_bound = single_cell_exp_params["Ad_sizer"] - 3* single_cell_exp_params["Ad_sizer_sigma"]*1.0136
    upper_bound = single_cell_exp_params["Ad_sizer"] + 3* single_cell_exp_params["Ad_sizer_sigma"]*1.0136
    Ad_sizer = r_truncnorm(single_cell_exp_params["Ad_sizer"],
                           single_cell_exp_params["Ad_sizer_sigma"],
                           lower_bound, upper_bound, 
                           size=n_div).astype(np.float64)
    
    return Ad_sizer

def compute_g_new_mu_new_scout(n_cells):
    g_mean = single_cell_exp_params["Gtime_mu_min"]
    g_sigma = single_cell_exp_params['Gtime_sigma_min']
    lower_bound = np.maximum(single_cell_exp_params["Gtime_min"],
                             g_mean - 3* g_sigma*1.0136)
    upper_bound = np.inf
    g_new = r_truncnorm(g_mean, g_sigma,
                        lower_bound, upper_bound, 
                        size=n_cells).astype(np.float64)
    
    mu_mean = single_cell_exp_params["alpha_mu_max"]
    mu_sigma = single_cell_exp_params['alpha_sigma_max']
    lower_bound = np.maximum(0.0,
                             mu_mean - 3* mu_sigma*1.0136)
    upper_bound = np.minimum(single_cell_exp_params["alpha_max"],
                             mu_mean + 3* mu_sigma*1.0136)
    mu_new = r_truncnorm(mu_mean, mu_sigma,
                         lower_bound, upper_bound, 
                         size=n_cells).astype(np.float64)
    
    return g_new, mu_new

## 3.3. Plot

In [ ]:
# Config
def set_mytheme_paper(ax):
    plt.rcParams["text.usetex"] = False
    plt.rcParams["font.family"] = "Helvetica"
    plt.rcParams["font.size"] = 7
    plt.rcParams["text.color"] = "black"
    mpl.rcParams['svg.fonttype'] = 'none'

    # title
    ax.title.set_fontsize(9.5)
    ax.title.set_color("black")
    ax.title.set_fontweight("bold")
    ax.title.set_position((0.5, 1.05))

    # axis
    ax.xaxis.label.set_size(8)
    ax.yaxis.label.set_size(8)
    ax.xaxis.label.set_color("black")
    ax.yaxis.label.set_color("black")

    # ticks
    ax.tick_params(axis='x', labelsize=6.5, colors="black")
    ax.tick_params(axis='y', labelsize=6.5, colors="black")

    # spine
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.0)

    # background
    ax.set_facecolor("none")
    ax.figure.set_facecolor("none")

    # grid
    ax.grid(False)

In [ ]:
# plot function
def plot_results_for_N_seaborn(fitted_params, name, n,
                               output_folder=None, fig_show=False):
    np.random.seed(42)
    
    # --- summarize to df ---
    df_lines = []
    df_box = []
    
    # labels
    legend_labels = {"obs": "Experimental data",
                     "sim": "Simulation data"}
    axis_labels = {"obs": "Experimental data\n(n=12)",
                   "sim": "Simulation data\n(n=12)"}
    type_colors = {legend_labels["obs"]: "salmon",
                   legend_labels["sim"]: "black",
                   axis_labels["obs"]: "salmon",
                   axis_labels["sim"]: "black"}
    
    for (key_name, key_n, key_id), vals in fitted_params.items():
        if key_name != name or key_n != n:
            continue
        (t_obs, N_obs, K_obs, 
         t_pred, N_hist, k_pred_mM, 
         B_hist, time_thresh_obs, time_thresh_pred) = vals
        # for line plot
        df_lines.append(pd.DataFrame({
            "Day": t_obs,
            "Nitrite": K_obs,
            "ID": f"ID{key_id}_obs",
            "Legend": legend_labels["obs"],
        }))
        df_lines.append(pd.DataFrame({
            "Day": t_pred,
            "Nitrite": k_pred_mM,
            "ID": f"ID{key_id}_sim",
            "Legend": legend_labels["sim"],
        }))
        # for box plot
        df_box.append({"AxisLabel": axis_labels["obs"], 
                       "Time": time_thresh_obs})
        df_box.append({"AxisLabel": axis_labels["sim"], 
                       "Time": time_thresh_pred})

    df_lines = pd.concat(df_lines, ignore_index=True)
    df_box = pd.DataFrame(df_box)

    # --- normalize time to reach threshold ---
    def normalize_or_dummy(group, axis_label):
        if group["Time"].notna().any():
            group["normalized_Time"] = group["Time"] / group["Time"].mean()
        else:
            group["normalized_Time"] = -1
        group["AxisLabel"] = axis_label
        return group
    df_box_valid = (
        df_box
        .groupby("AxisLabel", group_keys=False)
        .apply(lambda g: normalize_or_dummy(g, g.name), include_groups=False)
        .reset_index(drop=True)
    )

    # --- Figure 1: Nitrite lineplot ---
    fig, ax = plt.subplots(figsize=(3.2, 2.4))
    sns.lineplot(data=df_lines, x="Day", y="Nitrite", 
                 hue="Legend", style="Legend", units="ID",
                 markers=True, markeredgecolor="white", 
                 markersize=4.5, alpha=0.5, 
                 markeredgewidth=0.7, linewidth=1.4, dashes=False,
                 palette=type_colors, estimator=None)
    ax.set_xlabel("Time (day)")
    ax.set_ylabel("Nitrite (mM)")
    ax.legend(loc='upper left', frameon=False)
    set_mytheme_paper(ax)
    
    if output_folder:
        output_folder_lineplot = os.path.join(output_folder, "lineplots")
        os.makedirs(output_folder_lineplot, exist_ok=True)
        fig.savefig(os.path.join(output_folder_lineplot, f"Nitrite_lineplot_n{n}.png"), 
                    dpi=600, bbox_inches="tight")
        fig.savefig(os.path.join(output_folder_lineplot, f"Nitrite_lineplot_n{n}.svg"), 
                    format='svg', bbox_inches="tight")
        print(f'Saved to {os.path.join(output_folder_lineplot, f"Nitrite_lineplot_n{n}")} (.png & .svg)')
    if fig_show:
        plt.show()
    else:
        plt.close(fig)
        
    # --- Figure 2: Normalized threshold boxplot ---
    fig, ax = plt.subplots(figsize=(3.2, 2.4))
    sns.boxplot(data=df_box_valid, x="AxisLabel", y="normalized_Time",
                hue="AxisLabel", palette=type_colors,
                dodge=False, legend=False, ax=ax)
        
    # plot individual points (excluding -1 and NaN)
    df_nonan = df_box_valid[
        (df_box_valid["normalized_Time"].notna()) &
        (df_box_valid["normalized_Time"] != -1)
        ]
    sns.stripplot(data=df_nonan, x="AxisLabel", y="normalized_Time",
                  color="red", size=3.5, jitter=True, alpha=0.6, ax=ax)
    
    # plot × for -1 and NaN
    for i, label in enumerate(df_box_valid["AxisLabel"].unique()):
        mask = (
            ((df_box_valid["AxisLabel"] == label) & (df_box_valid["normalized_Time"] == -1)) |
            ((df_box_valid["AxisLabel"] == label) & (df_box_valid["normalized_Time"].isna()))
        )
        if mask.any():
            vals = df_box_valid.loc[df_box_valid["AxisLabel"] == label, "normalized_Time"]
            y_max = vals.max()
            if vals.eq(-1).all():
                y_max = 1.25
                ax.text(i, 1.0, "No awakening\nobserved",
                        ha="center", va="bottom", color="black",
                        fontsize=7, fontstyle="italic")
            n_points = mask.sum()

            # add jitter to x positions
            x_center = i
            jitter = np.random.uniform(-0.4, 0.4, size=n_points)  # adjust jitter range as needed
            x_pos = x_center + jitter
            y_pos = np.repeat(y_max*1.05, n_points)
            ax.scatter(x_pos, y_pos, 
                       marker="x", color="black", 
                       s=30, zorder=10)
    
    ax.set_xlabel("")
    ax.set_ylabel("Normalized time to reach 0.25 mM nitrite")
    ax.set_ylim(0.25, 2.0) 
    set_mytheme_paper(ax)
    
    if output_folder:
        output_folder_boxplot = os.path.join(output_folder, "boxplots")
        os.makedirs(output_folder_boxplot, exist_ok=True)
        fig.savefig(os.path.join(output_folder_boxplot, f"Nitrite_boxplot_n{n}.png"),
                    dpi=600, bbox_inches="tight")
        fig.savefig(os.path.join(output_folder_boxplot, f"Nitrite_boxplot_n{n}.svg"), 
                    format='svg', bbox_inches="tight")
        print(f'Saved to {os.path.join(output_folder_boxplot, f"Nitrite_boxplot_n{n}")} (.png & .svg)')
    if fig_show:
        plt.show()
    else:
        plt.close(fig)

## 3.3. Run

In [ ]:
def run_simulation(id, df, 
                   model, 
                   k0_mM_active, 
                   biomass_production_density0_active, 
                   N0, 
                   Nitrite_detection_threshold,
                   weibull_scale=None, weibull_shape=None,
                   IF_reserve=False):
    t_obs = df["Day"].values
    N_obs = df["cell_num"].values
    K_obs = df["Nitrite"].values
    
    # simulate stochastic pipetting
    N0_pipette = np.random.poisson(lam=N0, size=1).item()

    # run simulation
    ((t_pred, N_hist, k_pred_mM, B_hist), 
     cells_history_df, nondividing_cells_df) = (model
                                                (t_obs, N0_pipette, 
                                                 k0_mM_active, biomass_production_density0_active,
                                                 weibull_scale=weibull_scale, weibull_shape=weibull_shape,
                                                 IF_reserve = IF_reserve))
    
    # calculate time to reach threshold
    try:
        if K_obs.max() < Nitrite_detection_threshold:
            time_thresh_obs = np.nan
        else:
            time_thresh_obs = np.interp(Nitrite_detection_threshold, K_obs, t_obs)
        if k_pred_mM.max() < Nitrite_detection_threshold:
            time_thresh_pred = np.nan
        else:
            time_thresh_pred = np.interp(Nitrite_detection_threshold, k_pred_mM, t_pred)
    except Exception:
        time_thresh_obs, time_thresh_pred = np.nan, np.nan

    return (id, 
            (t_obs, N_obs, K_obs, 
             t_pred, N_hist, k_pred_mM,
             B_hist, time_thresh_obs, time_thresh_pred),
             cells_history_df, nondividing_cells_df)

In [ ]:
def reservoir_add(cell_info, T0_value, reservoirs, counts, k=1000):
    counts[T0_value] += 1
    n_seen = counts[T0_value]

    if len(reservoirs[T0_value]) < k:
        reservoirs[T0_value].append(cell_info)
    else:
        j = random.randint(0, n_seen - 1)
        if j < k:
            reservoirs[T0_value][j] = cell_info

## 3.4. Basic

In [ ]:
def simulate_basic_timer_sizer(t_obs, N0, 
                               k0_mM, biomass_production_density0,
                               weibull_scale=None, weibull_shape=None,
                               IF_reserve = False,
                               dt_hour=10.0, max_cells=int(5e7), max_records=1100):
    # --- time（hour） ---
    T_hours = int(np.ceil(float(t_obs[-1]) * 24.0))
    t_hours = np.arange(0.0, T_hours + dt_hour, dt_hour, dtype=float)
    t_day = t_hours / 24.0
    nT = t_hours.size

    # --- history arrays ---
    N_hist = np.empty(nT, dtype=float) # cells
    K_pmol_hist = np.empty(nT, dtype=float) # Nitrite, pmol
    B_hist = np.empty(nT, dtype=float) # ∆Vt, µm^3/mL
    
    # --- initial conditions ---
    N0 = int(N0)
    N_hist[0] = int(N0)
    k0_pM = k0_mM * 1e9 # mM -> pM
    k0_pmol = k0_pM * 1e-3 # pM -> pmol (volume 1e-3 L)
    K_pmol_hist[0] = k0_pmol
    B_hist[0] = float(biomass_production_density0)
    
    # --- history arrays for cells ---
    cells_age = np.zeros(max_cells, dtype=np.float64)
    cells_gtime = np.zeros(max_cells, dtype=np.float64)
    cells_mu = np.zeros(max_cells, dtype=np.float64)
    cells_volume = np.zeros(max_cells, dtype=np.float64)
    cells_volume_birth = np.zeros(max_cells, dtype=np.float64)
    cells_B_birth = np.zeros(max_cells, dtype=np.float64)
    cells_birth_time = np.zeros(max_cells, dtype=int)
    cells_sizer = np.zeros(max_cells, dtype=np.float64)
    cells_generation = np.zeros(max_cells, dtype=np.float64)
    
    # --- initial cell conditions ---
    A0 = rng.uniform(low=single_cell_exp_params["Ad_sizer"]/2,
                     high=single_cell_exp_params["Ad_sizer"],
                     size=N0)
    cells_volume[:N0] = A0 *0.75 # 0.75 µm, height of culturing chamber
    gtime0, mu0 = compute_g_new_mu_new(B_hist[0], cells_volume[:N0], N0)
    cells_age[:N0] = 0.0
    cells_gtime[:N0] = gtime0
    cells_mu[:N0] = mu0
    cells_volume_birth[:N0] = cells_volume[:N0]
    cells_B_birth[:N0] = B_hist[0]
    cells_birth_time[:N0] = 0
    cells_sizer[:N0] = compute_sizer(N0)
    cells_generation[:N0] = 0
    n_cells = int(N0)
    
    # --- reservoir for cell division records ---
    reservoirs = defaultdict(list)
    counts = defaultdict(int)
    
    # --- main simulation loop ---
    for k in range(1, nT):
        k_pM = K_pmol_hist[k-1] / 1e-3
        k_mM = k_pM / 1e9
        inhib = 1.0 / (1.0 + (k_mM / Ki)) # non-competitive inhibition model
        
        # --- process age ---
        cells_age[:n_cells] += dt_hour

        # --- process volume and biomass ---
        cells_volume_elongated = compute_elongation(cells_volume[:n_cells], cells_mu[:n_cells] *inhib, dt_hour)
        dB = np.maximum(0.0, cells_volume_elongated - cells_volume[:n_cells])
        cells_volume[:n_cells] = cells_volume_elongated
        B_hist[k] = B_hist[k-1] + dB.sum()
        
        # --- nitrite production ---
        dK = dB / convergence_cell_birth_volume * dK_per_cell
        K_pmol_hist[k] = K_pmol_hist[k-1] + dK.sum()
        if K_pmol_hist[k] > nitrite_detect_limit_pmol:
            # nitrite detection limit reached
            excess = K_pmol_hist[k] - nitrite_detect_limit_pmol
            fraction = 1 - excess / dK.sum()
            B_hist[k] = B_hist[k-1] + dB.sum() * fraction
            K_pmol_hist[k] = nitrite_detect_limit_pmol
            N_hist[k] = n_cells
            print(f"reached nitrite detection limit, day: {t_day[k]:.2f}, cell number: {n_cells}")
            # fill the rest of history with current values
            N_hist[k+1:] = n_cells
            K_pmol_hist[k+1:] = nitrite_detect_limit_pmol
            B_hist[k+1:] = B_hist[k]
            break

        # --- weibull hazard ---
        # non

        # --- division decision: cells that satisfy both age and volume conditions ---
        div_mask = (cells_age[:n_cells] >= cells_gtime[:n_cells]) & (cells_volume[:n_cells] >= cells_sizer[:n_cells])
        div_idx = np.nonzero(div_mask)[0]
        n_div = div_idx.size

        if n_div > 0:
            if IF_reserve == True:
                # --- save divided cell history ---
                for i in div_idx:
                    record = {
                        "age": cells_age[i],
                        "gtime": cells_gtime[i],
                        "mu": cells_mu[i],
                        "volume_division": cells_volume[i],
                        "volume_birth": cells_volume_birth[i],
                        "volume_sizer": cells_sizer[i],
                        "biomass_production_density_at_birth": cells_B_birth[i],
                        "T0": cells_birth_time[i],
                        "generation": cells_generation[i],
                    }
                    T0_value = cells_birth_time[i]  # birth day
                    reservoir_add(record, T0_value, reservoirs, counts, k=max_records)
            
            # --- process cell division ---
            # add daughter cells A (update parent cells)
            cells_age[div_idx] = 0.0
            # calculate new volumes
            parent_vol = cells_volume[div_idx].copy()
            divR = rng.normal(single_cell_exp_params["divR_mu"], single_cell_exp_params["divR_sigma"], size=n_div)
            volume_new_A = parent_vol * divR
            # calculate new generation time and elongation rate
            cells_gtime[div_idx], cells_mu[div_idx] = compute_g_new_mu_new(B_hist[k], volume_new_A, n_div)
            cells_volume[div_idx] = volume_new_A
            cells_volume_birth[div_idx] = volume_new_A
            cells_B_birth[div_idx] = B_hist[k]
            cells_birth_time[div_idx] = k*dt_hour
            cells_sizer[div_idx] = compute_sizer(n_div)
            cells_generation[div_idx] += 1
            
            # add daughter cells B
            new_start = n_cells
            new_end = n_cells + n_div
            if new_end > max_cells:
                print(f"reached nitrite detection limit, day: {t_day[k]:.2f}, cell number: {n_cells}")
                N_hist[k:] = n_cells
                K_pmol_hist[k:] = K_pmol_hist[k]
                B_hist[k:] = B_hist[k]
                break
            cells_age[new_start:new_end] = 0.0
            # calculate new volumes
            volume_new_B = parent_vol * (1-divR)
            # calculate new generation time and elongation rate
            cells_gtime[new_start:new_end], cells_mu[new_start:new_end] = compute_g_new_mu_new(B_hist[k], volume_new_B, n_div)
            cells_volume[new_start:new_end] = volume_new_B
            cells_volume_birth[new_start:new_end] = volume_new_B
            cells_B_birth[new_start:new_end] = B_hist[k]
            cells_birth_time[new_start:new_end] = k*dt_hour
            cells_sizer[new_start:new_end] = compute_sizer(n_div)
            cells_generation[new_start:new_end] = cells_generation[div_idx]
            
            n_cells = new_end
            
        N_hist[k] = n_cells
    
    K_mM_hist = K_pmol_hist / 1e-3 / 1e9
    
    # --- reserve for non-dividing cells ---
    reservoirs_2 = defaultdict(list)
    counts_2 = defaultdict(int)
    if IF_reserve == True:
            for i in range(n_cells):
                record_2 = {
                    "age": cells_age[i],
                    "gtime": cells_gtime[i],
                    "mu": cells_mu[i],
                    "volume_division": cells_volume[i],
                    "volume_birth": cells_volume_birth[i],
                    "volume_sizer": cells_sizer[i],
                    "biomass_production_density_at_birth": cells_B_birth[i],
                    "T0": cells_birth_time[i],
                    "generation": cells_generation[i],
                }
                T0_value = cells_birth_time[i]
                reservoir_add(record_2, T0_value, reservoirs_2, counts_2, k=max_records)
                
    del cells_age, cells_gtime, cells_mu, cells_volume, cells_volume_birth, cells_B_birth, cells_birth_time
    
    all_cells = []
    for _, cells in reservoirs.items():
        for cell_info in cells:
            all_cells.append(cell_info)
            
    all_cells_2 = []
    for _, cells in reservoirs_2.items():
        for cell_info in cells:
            all_cells_2.append(cell_info)
    
    if IF_reserve == True:
        reservoirs_df = pd.DataFrame(all_cells)
        reservoirs_df_2 = pd.DataFrame(all_cells_2)
    else:
        reservoirs_df = None
        reservoirs_df_2 = None
        
    del reservoirs, all_cells, reservoirs_2, all_cells_2

    return (t_day, N_hist, K_mM_hist, B_hist), reservoirs_df, reservoirs_df_2

## 3.5. Basic + weibull

In [ ]:
def simulate_basic_weibull(t_obs, N0, 
                           k0_mM, biomass_production_density0,
                           weibull_scale=None, weibull_shape=None,
                           IF_reserve = False,
                           dt_hour=10.0, max_cells=int(5e7), max_records=1100):
    # --- time（hour） ---
    T_hours = int(np.ceil(float(t_obs[-1]) * 24.0))
    t_hours = np.arange(0.0, T_hours + dt_hour, dt_hour, dtype=float)
    t_day = t_hours / 24.0
    nT = t_hours.size

   # --- history arrays ---
    N_hist = np.empty(nT, dtype=float) # cells
    K_pmol_hist = np.empty(nT, dtype=float) # Nitrite, pmol
    B_hist = np.empty(nT, dtype=float) # ∆Vt, µm^3/mL
    
    # --- initial conditions ---
    N0 = int(N0)
    N_hist[0] = int(N0)
    k0_pM = k0_mM * 1e9 # mM -> pM
    k0_pmol = k0_pM * 1e-3 # pM -> pmol (volume 1e-3 L)
    K_pmol_hist[0] = k0_pmol
    B_hist[0] = float(biomass_production_density0)
    rng = np.random.default_rng()  
    
    # --- history arrays for cells ---
    cells_age = np.zeros(max_cells, dtype=np.float64)
    cells_gtime = np.zeros(max_cells, dtype=np.float64)
    cells_mu = np.zeros(max_cells, dtype=np.float64)
    cells_volume = np.zeros(max_cells, dtype=np.float64)
    cells_volume_birth = np.zeros(max_cells, dtype=np.float64)
    cells_B_birth = np.zeros(max_cells, dtype=np.float64)
    cells_birth_time = np.zeros(max_cells, dtype=int)
    cells_sizer = np.zeros(max_cells, dtype=np.float64)
    cells_generation = np.zeros(max_cells, dtype=int)
    cells_flag = np.zeros(max_cells, dtype=bool) # flag for weibull awakening
    
    # --- initial cell conditions ---
    A0 = rng.uniform(low=single_cell_exp_params["Ad_sizer"]/2,
                     high=single_cell_exp_params["Ad_sizer"],
                     size=N0)
    cells_volume[:N0] = A0 *0.75 # 0.75 µm, height of culturing chamber
    gtime0, mu0 = compute_g_new_mu_new(B_hist[0], cells_volume[:N0], N0)
    cells_age[:N0] = 0.0    # hours
    cells_gtime[:N0] = gtime0
    cells_mu[:N0] = mu0
    cells_volume_birth[:N0] = cells_volume[:N0]
    cells_B_birth[:N0] = B_hist[0]
    cells_birth_time[:N0] = 0
    cells_sizer[:N0] = compute_sizer(N0)
    cells_generation[:N0] = 0
    cells_flag[:N0] = False
    n_cells = int(N0)
    
    # --- reservoir for cell division records ---
    reservoirs = defaultdict(list)
    counts = defaultdict(int)

    # --- main simulation loop ---
    for k in range(1, nT):
        k_pM = K_pmol_hist[k-1] / 1e-3
        k_mM = k_pM / 1e9
        inhib = 1.0 / (1.0 + (k_mM / Ki)) # non-competitive inhibition model
        
        # --- process age ---
        cells_age[:n_cells] += dt_hour
        
        # --- process volume and biomass ---
        cells_volume_elongated = compute_elongation(cells_volume[:n_cells], cells_mu[:n_cells] *inhib, dt_hour)
        dB = np.maximum(0.0, cells_volume_elongated - cells_volume[:n_cells])
        cells_volume[:n_cells] = cells_volume_elongated
        B_hist[k] = B_hist[k-1] + dB.sum()
        
        # --- nitrite production ---
        dK = dB / convergence_cell_birth_volume * dK_per_cell
        K_pmol_hist[k] = K_pmol_hist[k-1] + dK.sum()
        if K_pmol_hist[k] > nitrite_detect_limit_pmol:
            # nitrite detection limit reached
            excess = K_pmol_hist[k] - nitrite_detect_limit_pmol
            fraction = 1 - excess / dK.sum()
            B_hist[k] = B_hist[k-1] + dB.sum() * fraction
            K_pmol_hist[k] = nitrite_detect_limit_pmol
            N_hist[k] = n_cells
            print(f"reached nitrite detection limit, day: {t_day[k]:.2f}, cell number: {n_cells}")
            # fill the rest of history with current values
            N_hist[k+1:] = n_cells
            K_pmol_hist[k+1:] = nitrite_detect_limit_pmol
            B_hist[k+1:] = B_hist[k]
            break
        
        # --- weibull hazard ---
        haz = calculate_weibull_hazard_jit(cells_age[:n_cells], weibull_scale, weibull_shape)
        prob_dt = 1.0 - np.exp(-haz * dt_hour)
        scout_mask = (rng.random(n_cells) < prob_dt) & (~cells_flag[:n_cells])
        if np.any(scout_mask): 
            n_scout = np.nonzero(scout_mask)[0].size
            cells_gtime[:n_cells][scout_mask], cells_mu[:n_cells][scout_mask] = compute_g_new_mu_new_scout(n_scout)
            cells_flag[:n_cells][scout_mask] = True

        # --- division decision: cells that satisfy both age and volume conditions ---
        div_mask = (cells_age[:n_cells] >= cells_gtime[:n_cells]) & (cells_volume[:n_cells] >= cells_sizer[:n_cells])
        div_idx = np.nonzero(div_mask)[0]
        n_div = div_idx.size

        if n_div > 0:
            if IF_reserve == True:
                # --- save divided cell history ---
                for i in div_idx:
                    record = {
                        "age": cells_age[i],
                        "gtime": cells_gtime[i],
                        "mu": cells_mu[i],
                        "volume_division": cells_volume[i],
                        "volume_birth": cells_volume_birth[i],
                        "volume_sizer": cells_sizer[i],
                        "biomass_production_density_at_birth": cells_B_birth[i],
                        "T0": cells_birth_time[i],
                        "generation": cells_generation[i],
                    }
                    T0_value = cells_birth_time[i]  # birth day
                    reservoir_add(record, T0_value, reservoirs, counts, k=max_records)
                
            # --- process cell division ---
            # add daughter cells A (update parent cells)
            cells_age[div_idx] = 0.0
            # calculate new volumes
            parent_vol = cells_volume[div_idx].copy()
            divR = rng.normal(single_cell_exp_params["divR_mu"], single_cell_exp_params["divR_sigma"], size=n_div)
            volume_new_A = parent_vol * divR
            # calculate new generation time and elongation rate
            cells_gtime[div_idx], cells_mu[div_idx] = compute_g_new_mu_new(B_hist[k], volume_new_A, n_div)
            cells_volume[div_idx] = volume_new_A
            cells_volume_birth[div_idx] = volume_new_A
            cells_B_birth[div_idx] = B_hist[k]
            cells_birth_time[div_idx] = k*dt_hour
            cells_sizer[div_idx] = compute_sizer(n_div)
            cells_generation[div_idx] += 1
            cells_flag[div_idx] = cells_flag[div_idx] # inherit flag(continue active growth)
            
            # add daughter cells B
            new_start = n_cells
            new_end = n_cells + n_div
            if new_end > max_cells:
                print(f"reached nitrite detection limit, day: {t_day[k]:.2f}, cell number: {n_cells}")
                N_hist[k:] = n_cells
                K_pmol_hist[k:] = K_pmol_hist[k]
                B_hist[k:] = B_hist[k]
                break
            cells_age[new_start:new_end] = 0.0
            # calculate new volumes
            volume_new_B = parent_vol * (1-divR)
            # calculate new generation time and elongation rate
            cells_gtime[new_start:new_end], cells_mu[new_start:new_end] = compute_g_new_mu_new(B_hist[k], volume_new_B, n_div)
            cells_volume[new_start:new_end] = volume_new_B
            cells_volume_birth[new_start:new_end] = volume_new_B
            cells_B_birth[new_start:new_end] = B_hist[k]
            cells_birth_time[new_start:new_end] = k*dt_hour
            cells_sizer[new_start:new_end] = compute_sizer(n_div)
            cells_generation[new_start:new_end] = cells_generation[div_idx]
            cells_flag[new_start:new_end] = cells_flag[div_idx] # inherit flag(continue active growth)
            
            n_cells = new_end
            
            # --- update generation time & elongation rate of cell(flag=True) ---
            flag_idx = np.concatenate([div_idx, np.arange(new_start, new_end)])
            flag_idx = flag_idx[cells_flag[flag_idx]]
            if flag_idx.size > 0:
                n_scout = flag_idx.size
                cells_gtime[flag_idx], cells_mu[flag_idx] = compute_g_new_mu_new_scout(n_scout)
            
        N_hist[k] = n_cells
    
    K_mM_hist = K_pmol_hist / 1e-3 / 1e9
    
    # --- reserve for non-dividing cells ---
    reservoirs_2 = defaultdict(list)
    counts_2 = defaultdict(int)
    if IF_reserve == True:
            for i in range(n_cells):
                record_2 = {
                    "age": cells_age[i],
                    "gtime": cells_gtime[i],
                    "mu": cells_mu[i],
                    "volume_division": cells_volume[i],
                    "volume_birth": cells_volume_birth[i],
                    "volume_sizer": cells_sizer[i],
                    "biomass_production_density_at_birth": cells_B_birth[i],
                    "T0": cells_birth_time[i],
                    "generation": cells_generation[i],
                }
                T0_value = cells_birth_time[i]
                reservoir_add(record_2, T0_value, reservoirs_2, counts_2, k=max_records)
    
    del cells_age, cells_gtime, cells_mu, cells_volume, cells_volume_birth, cells_B_birth, cells_birth_time, cells_flag
    
    all_cells = []
    for _, cells in reservoirs.items():
        for cell_info in cells:
            all_cells.append(cell_info)
            
    all_cells_2 = []
    for _, cells in reservoirs_2.items():
        for cell_info in cells:
            all_cells_2.append(cell_info)
    
    if IF_reserve == True:
        reservoirs_df = pd.DataFrame(all_cells)
        reservoirs_df_2 = pd.DataFrame(all_cells_2)
    else:
        reservoirs_df = None
        reservoirs_df_2 = None
        
    del reservoirs, all_cells, reservoirs_2, all_cells_2
    
    return (t_day, N_hist, K_mM_hist, B_hist), reservoirs_df, reservoirs_df_2

# 4. Simulation

## 4.1. Basic

### 4.1.1. CFS+

In [ ]:
params_for_basic_CFS = {       
       "CFS_10^5": {
              "specie": "PY1",
              "cellDensity": "10^5",
              "init_cell_num":1e5,
              "k0_mM":[initial_nitrite_CFS_1,
                       initial_nitrite_CFS_2,
                       initial_nitrite_CFS_3],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[deltaVt0_CFS_1,
                                             deltaVt0_CFS_2,
                                             deltaVt0_CFS_3], 
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_timer_sizer
       },
       
       "CFS_10^3": {
              "specie": "PY1",
              "cellDensity": "10^3",
              "init_cell_num":1e3,
              "k0_mM":[initial_nitrite_CFS_1,
                       initial_nitrite_CFS_2,
                       initial_nitrite_CFS_3],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[deltaVt0_CFS_1,
                                             deltaVt0_CFS_2,
                                             deltaVt0_CFS_3], 
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_timer_sizer
       },
       
       "CFS_10^1": {
              "specie": "PY1",
              "cellDensity": "10^1",
              "init_cell_num":1e1,
              "k0_mM":[initial_nitrite_CFS_1,
                       initial_nitrite_CFS_2,
                       initial_nitrite_CFS_3],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[deltaVt0_CFS_1,
                                             deltaVt0_CFS_2,
                                             deltaVt0_CFS_3],  
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_timer_sizer
       },
       
       "CFS_10^1_lambdaAdjusted": {
              "specie": "PY1",
              "cellDensity": "10^1",
              "init_cell_num":[1e1,
                               -np.log(3/12), # 3 out of 12 wells did not show nitrite production(N=2)
                               -np.log(1/12) # 1 out of 12 wells did not show nitrite production(N=3)
                               ],
              "k0_mM":[initial_nitrite_CFS_1,
                       initial_nitrite_CFS_2,
                       initial_nitrite_CFS_3],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[deltaVt0_CFS_1,
                                             deltaVt0_CFS_2,
                                             deltaVt0_CFS_3],  
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_timer_sizer
       }
}

In [ ]:
for name, params in params_for_basic_CFS.items():
    for n_idx, n in enumerate(["1","2","3"]):
        print(f"=== {name} N={n} start simulation ===")
        
        init_cell_num = (
            params["init_cell_num"][n_idx]
            if name == "CFS_10^1_lambdaAdjusted"
            else params["init_cell_num"]
        )

        tasks = []
        IF_reserve=False
        for id in [str(i) for i in range(1,13)]:
            df = (exp_sup_mutate
                  .query("Specie == @params['specie'] and CellDensity == @params['cellDensity'] and N == @n and ID == @id and Nitrite < 1.5"))
            if not df.empty:
                tasks.append((id, df))
        results = Parallel(n_jobs=6)(delayed(run_simulation)(id, df, 
                                                             params["model"], 
                                                             params["k0_mM"][n_idx],
                                                             params["biomass_production_density0"][n_idx],
                                                             init_cell_num,
                                                             params['Nitrite_detection_threshold'],
                                                             IF_reserve=IF_reserve
                                                             )
                                                             for id, df in tasks)
        
        # results_dict
        results_dict = { (name, n, id): val 
                        for id, val, _, _ in results }
        # save folder
        output_folder = f"./result/basic/{name}"
        os.makedirs(output_folder, exist_ok=True)

        # save biomass_history
        biomass_history = []
        for key, vals in results_dict.items():
            name_, n_, id_ = key
            t_obs, N_obs, K_obs, t_pred, N_hist, k_pred_mM, B_hist, obs_time, pred_time = vals
            for i in range(len(B_hist)):
                biomass_history.append({
                    "name": name_,
                    "N_sim": n_,
                    "ID": id_,
                    "t_hour": 10.0 * i,
                    "B": B_hist[i]
                })
        if biomass_history:
            output_folder_2 = os.path.join(output_folder, "biomass_records")
            os.makedirs(output_folder_2, exist_ok=True)
            biomass_history_df = pd.DataFrame(biomass_history)
            biomass_history_df.to_csv(f"{output_folder_2}/biomass_records_n{n}.csv", index=True)

        # save history_records and nondividing_cell_records
        if IF_reserve == True:
            history_records = { (name, n, id): cells_history_df
                               for id, _, cells_history_df, _ in results }
            nondividing_cells_records = { (name, n, id): nondividing_cells_df
                                         for id, _, _, nondividing_cells_df in results }

            output_folder_3 = os.path.join(output_folder, "history_records")
            os.makedirs(output_folder_3, exist_ok=True)
            output_folder_4 = os.path.join(output_folder, "nondividing_cell_records")
            os.makedirs(output_folder_4, exist_ok=True)
            
            all_histories = []
            for key, df in history_records.items():
                name_, n_, id_ = key
                df = df.copy()
                df["name"] = name_
                df["N"] = n_
                df["ID"] = id_
                all_histories.append(df)
            if all_histories:
                all_histories_df = pd.concat(all_histories, ignore_index=True)
                all_histories_df.to_csv(f"{output_folder_3}/history_records_n{n}.csv", index=False)
                
            all_nondivided = []
            for key, df in nondividing_cells_records.items():
                name_, n_, id_ = key
                df = df.copy()
                df["name"] = name_
                df["N"] = n_
                df["ID"] = id_
                all_nondivided.append(df)
            if all_nondivided:
                all_nondivided_df = pd.concat(all_nondivided, ignore_index=True)
                all_nondivided_df.to_csv(f"{output_folder_4}/nondividing_cell_records_n{n}.csv", index=False)
        
        # plot results
        plot_results_for_N_seaborn(results_dict, name, n, 
                                   output_folder=output_folder, fig_show=False)

### 4.1.2. CFS- (initial ∆Vt=1)

In [ ]:
params_for_basic_noCFS = {
       "noCFS_10^5": {
              "specie": "PY1",
              "cellDensity": "10^5",
              "init_cell_num":1e5,
              "k0_mM":[0, 0, 0],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[1, 1, 1], 
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_timer_sizer
       },
       
       "noCFS_10^3": {
              "specie": "PY1",
              "cellDensity": "10^3",
              "init_cell_num":1e3,
              "k0_mM":[0, 0, 0],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[1, 1, 1],
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_timer_sizer
       },
       
       "noCFS_10^1": {
              "specie": "PY1",
              "cellDensity": "10^1",
              "init_cell_num":1e1,
              "k0_mM":[0, 0, 0],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[1, 1, 1],
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_timer_sizer
       }
}

In [ ]:
for name, params in params_for_basic_noCFS.items():
    for n_idx, n in enumerate(["1","2","3"]):
        print(f"=== {name} N={n} start simulation ===")

        tasks = []
        IF_reserve=False
        for id in [str(i) for i in range(1,13)]:
            df = (exp_noSup_mutate.
                  query("Specie == @params['specie'] and CellDensity == @params['cellDensity'] and N == @n and ID == @id and Nitrite < 1.5"))
            if not df.empty:
                tasks.append((id, df))
        results = Parallel(n_jobs=6)(delayed(run_simulation)(id, df,
                                                             params["model"], 
                                                             params["k0_mM"][n_idx],
                                                             params["biomass_production_density0"][n_idx],
                                                             params["init_cell_num"],
                                                             params['Nitrite_detection_threshold'],
                                                             IF_reserve=IF_reserve
                                                             )
                                                             for id, df in tasks)
        
        # results_dict
        results_dict = { (name, n, id): val 
                        for id, val, _, _ in results }
        # save folder
        output_folder = f"./result/basic/{name}"
        os.makedirs(output_folder, exist_ok=True)

        # save biomass_history
        biomass_history = []
        for key, vals in results_dict.items():
            name_, n_, id_ = key
            t_obs, N_obs, K_obs, t_pred, N_hist, k_pred_mM, B_hist, obs_time, pred_time = vals
            for i in range(len(B_hist)):
                biomass_history.append({
                    "name": name_,
                    "N_sim": n_,
                    "ID": id_,
                    "t_hour": 10.0 * i,
                    "B": B_hist[i]
                })
        if biomass_history:
            output_folder_2 = os.path.join(output_folder, "biomass_records")
            os.makedirs(output_folder_2, exist_ok=True)
            biomass_history_df = pd.DataFrame(biomass_history)
            biomass_history_df.to_csv(f"{output_folder_2}/biomass_records_n{n}.csv", index=True)

        # save history_records and nondividing_cell_records
        if IF_reserve == True:
            history_records = { (name, n, id): cells_history_df
                               for id, _, cells_history_df, _ in results }
            nondividing_cells_records = { (name, n, id): nondividing_cells_df
                                         for id, _, _, nondividing_cells_df in results }

            output_folder_3 = os.path.join(output_folder, "history_records")
            os.makedirs(output_folder_3, exist_ok=True)
            output_folder_4 = os.path.join(output_folder, "nondividing_cell_records")
            os.makedirs(output_folder_4, exist_ok=True)
            
            all_histories = []
            for key, df in history_records.items():
                name_, n_, id_ = key
                df = df.copy()
                df["name"] = name_
                df["N"] = n_
                df["ID"] = id_
                all_histories.append(df)
            if all_histories:
                all_histories_df = pd.concat(all_histories, ignore_index=True)
                all_histories_df.to_csv(f"{output_folder_3}/history_records_n{n}.csv", index=False)
                
            all_nondivided = []
            for key, df in nondividing_cells_records.items():
                name_, n_, id_ = key
                df = df.copy()
                df["name"] = name_
                df["N"] = n_
                df["ID"] = id_
                all_nondivided.append(df)
            if all_nondivided:
                all_nondivided_df = pd.concat(all_nondivided, ignore_index=True)
                all_nondivided_df.to_csv(f"{output_folder_4}/nondividing_cell_records_n{n}.csv", index=False)
        
        # plot results
        plot_results_for_N_seaborn(results_dict, name, n, 
                                   output_folder=output_folder, fig_show=False)

### 4.1.3. CFS- (Re-define initial ∆Vt)

In [ ]:
params_for_basic_noCFS_new_deltaVt = {
       "noCFS_10^5_new_deltaVt": {
              "specie": "PY1",
              "cellDensity": "10^5",
              "init_cell_num":1e5,
              "k0_mM":[0, 0, 0],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[deltaVt0_noCFS_10_5,
                                             deltaVt0_noCFS_10_5,
                                             deltaVt0_noCFS_10_5], 
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_timer_sizer
       },
       
       "noCFS_10^3_new_deltaVt": {
              "specie": "PY1",
              "cellDensity": "10^3",
              "init_cell_num":1e3,
              "k0_mM":[0, 0, 0],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[deltaVt0_noCFS_10_3,
                                             deltaVt0_noCFS_10_3,
                                             deltaVt0_noCFS_10_3], 
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_timer_sizer
       },
       
       "noCFS_10^1_new_deltaVt": {
              "specie": "PY1",
              "cellDensity": "10^1",
              "init_cell_num":1e1,
              "k0_mM":[0, 0, 0],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[deltaVt0_noCFS_10_1,
                                             deltaVt0_noCFS_10_1,
                                             deltaVt0_noCFS_10_1], 
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_timer_sizer
       }
}

In [ ]:
for name, params in params_for_basic_noCFS_new_deltaVt.items():
    for n_idx, n in enumerate(["1","2","3"]):
        print(f"=== {name} N={n} start simulation ===")

        tasks = []
        IF_reserve=False
        for id in [str(i) for i in range(1,13)]:
            df = (exp_noSup_mutate.
                  query("Specie == @params['specie'] and CellDensity == @params['cellDensity'] and N == @n and ID == @id and Nitrite < 1.5"))
            if not df.empty:
                tasks.append((id, df))
        results = Parallel(n_jobs=6)(delayed(run_simulation)(id, df,
                                                             params["model"], 
                                                             params["k0_mM"][n_idx],
                                                             params["biomass_production_density0"][n_idx],
                                                             params["init_cell_num"],
                                                             params['Nitrite_detection_threshold'],
                                                             IF_reserve=IF_reserve
                                                             )
                                                             for id, df in tasks)
        
        # results_dict
        results_dict = { (name, n, id): val 
                        for id, val, _, _ in results }
        # save folder
        output_folder = f"./result/basic/{name}"
        os.makedirs(output_folder, exist_ok=True)

        # save biomass_history
        biomass_history = []
        for key, vals in results_dict.items():
            name_, n_, id_ = key
            t_obs, N_obs, K_obs, t_pred, N_hist, k_pred_mM, B_hist, obs_time, pred_time = vals
            for i in range(len(B_hist)):
                biomass_history.append({
                    "name": name_,
                    "N_sim": n_,
                    "ID": id_,
                    "t_hour": 10.0 * i,
                    "B": B_hist[i]
                })
        if biomass_history:
            output_folder_2 = os.path.join(output_folder, "biomass_records")
            os.makedirs(output_folder_2, exist_ok=True)
            biomass_history_df = pd.DataFrame(biomass_history)
            biomass_history_df.to_csv(f"{output_folder_2}/biomass_records_n{n}.csv", index=True)

        # save history_records and nondividing_cell_records
        if IF_reserve == True:
            history_records = { (name, n, id): cells_history_df
                               for id, _, cells_history_df, _ in results }
            nondividing_cells_records = { (name, n, id): nondividing_cells_df
                                         for id, _, _, nondividing_cells_df in results }

            output_folder_3 = os.path.join(output_folder, "history_records")
            os.makedirs(output_folder_3, exist_ok=True)
            output_folder_4 = os.path.join(output_folder, "nondividing_cell_records")
            os.makedirs(output_folder_4, exist_ok=True)
            
            all_histories = []
            for key, df in history_records.items():
                name_, n_, id_ = key
                df = df.copy()
                df["name"] = name_
                df["N"] = n_
                df["ID"] = id_
                all_histories.append(df)
            if all_histories:
                all_histories_df = pd.concat(all_histories, ignore_index=True)
                all_histories_df.to_csv(f"{output_folder_3}/history_records_n{n}.csv", index=False)
                
            all_nondivided = []
            for key, df in nondividing_cells_records.items():
                name_, n_, id_ = key
                df = df.copy()
                df["name"] = name_
                df["N"] = n_
                df["ID"] = id_
                all_nondivided.append(df)
            if all_nondivided:
                all_nondivided_df = pd.concat(all_nondivided, ignore_index=True)
                all_nondivided_df.to_csv(f"{output_folder_4}/nondividing_cell_records_n{n}.csv", index=False)
        
        # plot results
        plot_results_for_N_seaborn(results_dict, name, n, 
                                   output_folder=output_folder, fig_show=False)

## 4.2. Basic(numerous)

### 4.2.1. CFS+

In [ ]:
for name, params in params_for_basic_CFS.items():
    for n in range(n_repeat):
        print(f"=== {name} N={n} start simulation ===")
        
        n_idx = 2
        init_cell_num = (
            params["init_cell_num"][n_idx]
            if name == "CFS_10^1_lambdaAdjusted"
            else params["init_cell_num"]
        )

        tasks = []
        IF_reserve=False # Fix
        for id in [str(i) for i in range(1,13)]:
            df = (exp_sup_mutate
                  .query("Specie == @params['specie'] and CellDensity == @params['cellDensity'] and N == '1' and ID == @id and Nitrite < 1.5"))
            if not df.empty:
                tasks.append((id, df))
        results = Parallel(n_jobs=6)(delayed(run_simulation)(id, df,
                                                             params["model"], 
                                                             params["k0_mM"][1], # use n=1 initial nitrite as representative
                                                             params["biomass_production_density0"][1], # use n=1 initial biomass as representative
                                                             init_cell_num,
                                                             params['Nitrite_detection_threshold'],
                                                             IF_reserve=IF_reserve
                                                             )
                                                             for id, df in tasks)
        
        # results_dict
        results_dict = { (name, n, id): val 
                        for id, val, _, _ in results }
        # save folder
        output_folder = f"./result/basic_numerous/{name}"
        os.makedirs(output_folder, exist_ok=True)

        # save biomass_history
        biomass_history = []
        for key, vals in results_dict.items():
            name_, n_, id_ = key
            t_obs, N_obs, K_obs, t_pred, N_hist, k_pred_mM, B_hist, obs_time, pred_time = vals
            for i in range(len(B_hist)):
                biomass_history.append({
                    "name": name_,
                    "N_sim": n_,
                    "ID": id_,
                    "t_hour": 10.0 * i,
                    "B": B_hist[i]
                })
        if biomass_history:
            output_folder_2 = os.path.join(output_folder, "biomass_records")
            os.makedirs(output_folder_2, exist_ok=True)
            biomass_history_df = pd.DataFrame(biomass_history)
            biomass_history_df.to_csv(f"{output_folder_2}/biomass_records_n{n}.csv", index=True)


### 4.2.2. CFS- (initial ∆Vt=1)

In [ ]:
for name, params in params_for_basic_noCFS.items():
    for n in range(n_repeat):
        print(f"=== {name} N={n} start simulation ===")
        
        tasks = []
        IF_reserve=False # Fix
        for id in [str(i) for i in range(1,13)]:
            df = (exp_noSup_mutate
                  .query("Specie == @params['specie'] and CellDensity == @params['cellDensity'] and N == '1' and ID == @id and Nitrite < 1.5"))
            if not df.empty:
                tasks.append((id, df))
        results = Parallel(n_jobs=6)(delayed(run_simulation)(id, df,
                                                             params["model"], 
                                                             params["k0_mM"][1], # use n=1 initial nitrite as representative
                                                             params["biomass_production_density0"][1], # use n=1 initial biomass as representative
                                                             params["init_cell_num"],
                                                             params['Nitrite_detection_threshold'],
                                                             IF_reserve=IF_reserve
                                                             )
                                                             for id, df in tasks)
        
        # results_dict
        results_dict = { (name, n, id): val 
                        for id, val, _, _ in results }
        # save folder
        output_folder = f"./result/basic_numerous/{name}"
        os.makedirs(output_folder, exist_ok=True)

        # save biomass_history
        biomass_history = []
        for key, vals in results_dict.items():
            name_, n_, id_ = key
            t_obs, N_obs, K_obs, t_pred, N_hist, k_pred_mM, B_hist, obs_time, pred_time = vals
            for i in range(len(B_hist)):
                biomass_history.append({
                    "name": name_,
                    "N_sim": n_,
                    "ID": id_,
                    "t_hour": 10.0 * i,
                    "B": B_hist[i]
                })
        if biomass_history:
            output_folder_2 = os.path.join(output_folder, "biomass_records")
            os.makedirs(output_folder_2, exist_ok=True)
            biomass_history_df = pd.DataFrame(biomass_history)
            biomass_history_df.to_csv(f"{output_folder_2}/biomass_records_n{n}.csv", index=True)

### 4.2.3. CFS- (Re-define initial ∆Vt)

In [ ]:
for name, params in params_for_basic_noCFS_new_deltaVt.items():
    for n in range(n_repeat):
        print(f"=== {name} N={n} start simulation ===")
        
        tasks = []
        IF_reserve=False # Fix
        for id in [str(i) for i in range(1,13)]:
            df = (exp_noSup_mutate
                  .query("Specie == @params['specie'] and CellDensity == @params['cellDensity'] and N == '1' and ID == @id and Nitrite < 1.5"))
            if not df.empty:
                tasks.append((id, df))
        results = Parallel(n_jobs=6)(delayed(run_simulation)(id, df,
                                                             params["model"], 
                                                             params["k0_mM"][1], # use n=1 initial nitrite as representative
                                                             params["biomass_production_density0"][1], # use n=1 initial biomass as representative
                                                             params["init_cell_num"],
                                                             params['Nitrite_detection_threshold'],
                                                             IF_reserve=IF_reserve
                                                             )
                                                             for id, df in tasks)
        
        # results_dict
        results_dict = { (name, n, id): val 
                        for id, val, _, _ in results }
        # save folder
        output_folder = f"./result/basic_numerous/{name}"
        os.makedirs(output_folder, exist_ok=True)

        # save biomass_history
        biomass_history = []
        for key, vals in results_dict.items():
            name_, n_, id_ = key
            t_obs, N_obs, K_obs, t_pred, N_hist, k_pred_mM, B_hist, obs_time, pred_time = vals
            for i in range(len(B_hist)):
                biomass_history.append({
                    "name": name_,
                    "N_sim": n_,
                    "ID": id_,
                    "t_hour": 10.0 * i,
                    "B": B_hist[i]
                })
        if biomass_history:
            output_folder_2 = os.path.join(output_folder, "biomass_records")
            os.makedirs(output_folder_2, exist_ok=True)
            biomass_history_df = pd.DataFrame(biomass_history)
            biomass_history_df.to_csv(f"{output_folder_2}/biomass_records_n{n}.csv", index=True)

## 4.3. weibull

### 4.3.1. CFS- (Re-define initial ∆Vt)

In [ ]:
params_for_weibull_noCFS = {
       "noCFS_10^5_new_deltaVt": {
              "specie": "PY1",
              "cellDensity": "10^5",
              "init_cell_num":1e5,
              "k0_mM":[0, 0, 0],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[deltaVt0_noCFS_10_5,
                                             deltaVt0_noCFS_10_5,
                                             deltaVt0_noCFS_10_5], 
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_weibull,
              "weibull_scale":4314.262685902414,
              "weibull_shape":2.7854412942326037,
       },
       
       "noCFS_10^3_new_deltaVt": {
              "specie": "PY1",
              "cellDensity": "10^3",
              "init_cell_num":1e3,
              "k0_mM":[0, 0, 0],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[deltaVt0_noCFS_10_3,
                                             deltaVt0_noCFS_10_3,
                                             deltaVt0_noCFS_10_3], 
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_weibull,
              "weibull_scale":22478.429638231824,
              "weibull_shape":2.252773302589696,
       },
       
       "noCFS_10^1_new_deltaVt": {
              "specie": "PY1",
              "cellDensity": "10^1",
              "init_cell_num":1e1,
              "k0_mM":[0, 0, 0],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[deltaVt0_noCFS_10_1,
                                             deltaVt0_noCFS_10_1,
                                             deltaVt0_noCFS_10_1],  
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_weibull,
              "weibull_scale":22478.429638231824,
              "weibull_shape":2.252773302589696,
       },
}

In [ ]:
for name, params in params_for_weibull_noCFS.items():
    for n_idx, n in enumerate(["1","2","3"]):
        print(f"=== {name} N={n} start simulation ===")
        
        tasks = []
        IF_reserve=False
        for id in [str(i) for i in range(1,13)]:
            df = (exp_noSup_mutate
                  .query("Specie == @params['specie'] and CellDensity == @params['cellDensity'] and N == @n and ID == @id and Nitrite < 1.5"))
            if not df.empty:
                tasks.append((id, df))
        results = Parallel(n_jobs=6)(delayed(run_simulation)(id, df,
                                                             params["model"], 
                                                             params["k0_mM"][n_idx],
                                                             params["biomass_production_density0"][n_idx],
                                                             params["init_cell_num"],
                                                             params['Nitrite_detection_threshold'],
                                                             weibull_scale=params["weibull_scale"],
                                                             weibull_shape=params["weibull_shape"],
                                                             IF_reserve=IF_reserve
                                                             )
                                                             for id, df in tasks)
        
        # results_dict
        results_dict = { (name, n, id): val 
                        for id, val, _, _ in results }
        # save folder
        output_folder = f"./result/weibull/{name}"
        os.makedirs(output_folder, exist_ok=True)

        # save biomass_history
        biomass_history = []
        for key, vals in results_dict.items():
            name_, n_, id_ = key
            t_obs, N_obs, K_obs, t_pred, N_hist, k_pred_mM, B_hist, obs_time, pred_time = vals
            for i in range(len(B_hist)):
                biomass_history.append({
                    "name": name_,
                    "N_sim": n_,
                    "ID": id_,
                    "t_hour": 10.0 * i,
                    "B": B_hist[i]
                })
        if biomass_history:
            output_folder_2 = os.path.join(output_folder, "biomass_records")
            os.makedirs(output_folder_2, exist_ok=True)
            biomass_history_df = pd.DataFrame(biomass_history)
            biomass_history_df.to_csv(f"{output_folder_2}/biomass_records_n{n}.csv", index=True)

        # save history_records and nondividing_cell_records
        if IF_reserve == True:
            history_records = { (name, n, id): cells_history_df
                               for id, _, cells_history_df, _ in results }
            nondividing_cells_records = { (name, n, id): nondividing_cells_df
                                         for id, _, _, nondividing_cells_df in results }

            output_folder_3 = os.path.join(output_folder, "history_records")
            os.makedirs(output_folder_3, exist_ok=True)
            output_folder_4 = os.path.join(output_folder, "nondividing_cell_records")
            os.makedirs(output_folder_4, exist_ok=True)
            
            all_histories = []
            for key, df in history_records.items():
                name_, n_, id_ = key
                df = df.copy()
                df["name"] = name_
                df["N"] = n_
                df["ID"] = id_
                all_histories.append(df)
            if all_histories:
                all_histories_df = pd.concat(all_histories, ignore_index=True)
                all_histories_df.to_csv(f"{output_folder_3}/history_records_n{n}.csv", index=False)
                
            all_nondivided = []
            for key, df in nondividing_cells_records.items():
                name_, n_, id_ = key
                df = df.copy()
                df["name"] = name_
                df["N"] = n_
                df["ID"] = id_
                all_nondivided.append(df)
            if all_nondivided:
                all_nondivided_df = pd.concat(all_nondivided, ignore_index=True)
                all_nondivided_df.to_csv(f"{output_folder_4}/nondividing_cell_records_n{n}.csv", index=False)
        
        # plot results
        plot_results_for_N_seaborn(results_dict, name, n, 
                                   output_folder=output_folder, fig_show=False)

### 4.3.2. CFS+

In [ ]:
params_for_weibull_CFS = {       
       "CFS_10^5": {
              "specie": "PY1",
              "cellDensity": "10^5",
              "init_cell_num":1e5,
              "k0_mM":[initial_nitrite_CFS_1,
                       initial_nitrite_CFS_2,
                       initial_nitrite_CFS_3],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[deltaVt0_CFS_1,
                                             deltaVt0_CFS_2,
                                             deltaVt0_CFS_3], 
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_weibull,
              "weibull_scale":4314.262685902414,
              "weibull_shape":2.7854412942326037,
       },
       
       "CFS_10^3": {
              "specie": "PY1",
              "cellDensity": "10^3",
              "init_cell_num":1e3,
              "k0_mM":[initial_nitrite_CFS_1,
                       initial_nitrite_CFS_2,
                       initial_nitrite_CFS_3],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[deltaVt0_CFS_1,
                                             deltaVt0_CFS_2,
                                             deltaVt0_CFS_3], 
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_weibull,
              "weibull_scale":4314.262685902414,
              "weibull_shape":2.7854412942326037,
       },
       
       "CFS_10^1": {
              "specie": "PY1",
              "cellDensity": "10^1",
              "init_cell_num":1e1,
              "k0_mM":[initial_nitrite_CFS_1,
                       initial_nitrite_CFS_2,
                       initial_nitrite_CFS_3],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[deltaVt0_CFS_1,
                                             deltaVt0_CFS_2,
                                             deltaVt0_CFS_3],  
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_weibull,
              "weibull_scale":4314.262685902414,
              "weibull_shape":2.7854412942326037,
       },
       
       "CFS_10^1_lambdaAdjusted": {
              "specie": "PY1",
              "cellDensity": "10^1",
              "init_cell_num":[1e1,
                               -np.log(3/12), # 3 out of 12 wells did not show nitrite production(N=2)
                               -np.log(1/12) # 1 out of 12 wells did not show nitrite production(N=3)
                               ],
              "k0_mM":[initial_nitrite_CFS_1,
                       initial_nitrite_CFS_2,
                       initial_nitrite_CFS_3],
              "Nitrite_detection_threshold":0.25,
              "biomass_production_density0":[deltaVt0_CFS_1,
                                             deltaVt0_CFS_2,
                                             deltaVt0_CFS_3],  
              "x1": "Day",
              "x2": "cell_num",
              "model": simulate_basic_weibull,
              "weibull_scale":4314.262685902414,
              "weibull_shape":2.7854412942326037,
       }
}

In [ ]:
for name, params in params_for_weibull_CFS.items():
    for n_idx, n in enumerate(["1","2","3"]):
        print(f"=== {name} N={n} start simulation ===")
            
        init_cell_num = (
            params["init_cell_num"][n_idx]
            if name == "CFS_10^1_lambdaAdjusted"
            else params["init_cell_num"]
        )

        tasks = []
        IF_reserve=False
        for id in [str(i) for i in range(1,13)]:
            df = (exp_sup_mutate
                  .query("Specie == @params['specie'] and CellDensity == @params['cellDensity'] and N == @n and ID == @id and Nitrite < 1.5"))
            if not df.empty:
                tasks.append((id, df))
        results = Parallel(n_jobs=6)(delayed(run_simulation)(id, df, 
                                                             params["model"], 
                                                             params["k0_mM"][n_idx],
                                                             params["biomass_production_density0"][n_idx],
                                                             init_cell_num,
                                                             params['Nitrite_detection_threshold'],
                                                             weibull_scale=params["weibull_scale"],
                                                             weibull_shape=params["weibull_shape"],
                                                             IF_reserve=IF_reserve
                                                             )
                                                             for id, df in tasks)
        
        # results_dict
        results_dict = { (name, n, id): val 
                        for id, val, _, _ in results }
        # save folder
        output_folder = f"./result/weibull/{name}"
        os.makedirs(output_folder, exist_ok=True)

        # save biomass_history
        biomass_history = []
        for key, vals in results_dict.items():
            name_, n_, id_ = key
            t_obs, N_obs, K_obs, t_pred, N_hist, k_pred_mM, B_hist, obs_time, pred_time = vals
            for i in range(len(B_hist)):
                biomass_history.append({
                    "name": name_,
                    "N_sim": n_,
                    "ID": id_,
                    "t_hour": 10.0 * i,
                    "B": B_hist[i]
                })
        if biomass_history:
            output_folder_2 = os.path.join(output_folder, "biomass_records")
            os.makedirs(output_folder_2, exist_ok=True)
            biomass_history_df = pd.DataFrame(biomass_history)
            biomass_history_df.to_csv(f"{output_folder_2}/biomass_records_n{n}.csv", index=True)

        # save history_records and nondividing_cell_records
        if IF_reserve == True:
            history_records = { (name, n, id): cells_history_df
                               for id, _, cells_history_df, _ in results }
            nondividing_cells_records = { (name, n, id): nondividing_cells_df
                                         for id, _, _, nondividing_cells_df in results }

            output_folder_3 = os.path.join(output_folder, "history_records")
            os.makedirs(output_folder_3, exist_ok=True)
            output_folder_4 = os.path.join(output_folder, "nondividing_cell_records")
            os.makedirs(output_folder_4, exist_ok=True)
            
            all_histories = []
            for key, df in history_records.items():
                name_, n_, id_ = key
                df = df.copy()
                df["name"] = name_
                df["N"] = n_
                df["ID"] = id_
                all_histories.append(df)
            if all_histories:
                all_histories_df = pd.concat(all_histories, ignore_index=True)
                all_histories_df.to_csv(f"{output_folder_3}/history_records_n{n}.csv", index=False)
                
            all_nondivided = []
            for key, df in nondividing_cells_records.items():
                name_, n_, id_ = key
                df = df.copy()
                df["name"] = name_
                df["N"] = n_
                df["ID"] = id_
                all_nondivided.append(df)
            if all_nondivided:
                all_nondivided_df = pd.concat(all_nondivided, ignore_index=True)
                all_nondivided_df.to_csv(f"{output_folder_4}/nondividing_cell_records_n{n}.csv", index=False)
        
        # plot results
        plot_results_for_N_seaborn(results_dict, name, n, 
                                   output_folder=output_folder, fig_show=False)

## 4.4. weibull(numerous)

### 4.4.1. CFS-

In [ ]:
for name, params in params_for_weibull_noCFS.items():
    for n in range(n_repeat):
        print(f"=== {name} N={n} start simulation ===")

        tasks = []
        IF_reserve=False # Fix
        for id in [str(i) for i in range(1,13)]:
            df = (exp_noSup_mutate
                  .query("Specie == @params['specie'] and CellDensity == @params['cellDensity'] and N == '1' and ID == @id and Nitrite < 1.5"))
            if not df.empty:
                tasks.append((id, df))
        results = Parallel(n_jobs=6)(delayed(run_simulation)(id, df,
                                                             params["model"], 
                                                             params["k0_mM"][1], # use n=1 initial nitrite as representative
                                                             params["biomass_production_density0"][1], # use n=1 initial biomass as representative
                                                             params["init_cell_num"],
                                                             params['Nitrite_detection_threshold'],
                                                             weibull_scale=params["weibull_scale"],
                                                             weibull_shape=params["weibull_shape"],
                                                             IF_reserve=IF_reserve
                                                             ) 
                                                             for id, df in tasks)
        # results_dict
        results_dict = { (name, n, id): val 
                        for id, val, _, _ in results }
        # save folder
        output_folder = f"./result/weibull_numerous/{name}"
        os.makedirs(output_folder, exist_ok=True)

        # save biomass_history
        biomass_history = []
        for key, vals in results_dict.items():
            name_, n_, id_ = key
            t_obs, N_obs, K_obs, t_pred, N_hist, k_pred_mM, B_hist, obs_time, pred_time = vals
            for i in range(len(B_hist)):
                biomass_history.append({
                    "name": name_,
                    "N_sim": n_,
                    "ID": id_,
                    "t_hour": 10.0 * i,
                    "B": B_hist[i]
                })
        if biomass_history:
            output_folder_2 = os.path.join(output_folder, "biomass_records")
            os.makedirs(output_folder_2, exist_ok=True)
            biomass_history_df = pd.DataFrame(biomass_history)
            biomass_history_df.to_csv(f"{output_folder_2}/biomass_records_n{n}.csv", index=True)
        

### 4.4.2. CFS+

In [ ]:
for name, params in params_for_weibull_CFS.items():
    for n in range(n_repeat):
        print(f"=== {name} N={n} start simulation ===")
        
        n_idx = 2
        init_cell_num = (
            params["init_cell_num"][n_idx]
            if name == "CFS_10^1_lambdaAdjusted"
            else params["init_cell_num"]
        )

        tasks = []
        IF_reserve=False # Fix
        for id in [str(i) for i in range(1,13)]:
            df = (exp_sup_mutate
                  .query("Specie == @params['specie'] and CellDensity == @params['cellDensity'] and N == '1' and ID == @id and Nitrite < 1.5"))
            if not df.empty:
                tasks.append((id, df))
        results = Parallel(n_jobs=6)(delayed(run_simulation)(id, df,
                                                             params["model"], 
                                                             params["k0_mM"][1], # use n=1 initial nitrite as representative
                                                             params["biomass_production_density0"][1], # use n=1 initial biomass as representative
                                                             init_cell_num,
                                                             params['Nitrite_detection_threshold'],
                                                             weibull_scale=params["weibull_scale"],
                                                             weibull_shape=params["weibull_shape"],
                                                             IF_reserve=IF_reserve
                                                             ) 
                                                             for id, df in tasks)
        # results_dict
        results_dict = { (name, n, id): val 
                        for id, val, _, _ in results }
        # save folder
        output_folder = f"./result/weibull_numerous/{name}"
        os.makedirs(output_folder, exist_ok=True)

        # save biomass_history
        biomass_history = []
        for key, vals in results_dict.items():
            name_, n_, id_ = key
            t_obs, N_obs, K_obs, t_pred, N_hist, k_pred_mM, B_hist, obs_time, pred_time = vals
            for i in range(len(B_hist)):
                biomass_history.append({
                    "name": name_,
                    "N_sim": n_,
                    "ID": id_,
                    "t_hour": 10.0 * i,
                    "B": B_hist[i]
                })
        if biomass_history:
            output_folder_2 = os.path.join(output_folder, "biomass_records")
            os.makedirs(output_folder_2, exist_ok=True)
            biomass_history_df = pd.DataFrame(biomass_history)
            biomass_history_df.to_csv(f"{output_folder_2}/biomass_records_n{n}.csv", index=True)
        

# 5. Image concatenate

In [ ]:
prefix = "./result/weibull"
name = "lineplots/Nitrite_lineplot_n1.svg"
output_folder = "../2_graph/result"
pt_to_mm = 25.4 / 72

Figure(
    "250mm", "125mm",
    SVG(os.path.join(prefix, "CFS_10^5", name)).scale(pt_to_mm).move(0, 0),
    SVG(os.path.join(prefix, "CFS_10^3", name)).scale(pt_to_mm).move(3.2*25.4, 0),
    SVG(os.path.join(prefix, "CFS_10^1", name)).scale(pt_to_mm).move(3.2*25.4*2, 0),
    SVG(os.path.join(prefix, "noCFS_10^5_new_deltaVt", name)).scale(pt_to_mm).move(0, 2.4*25.4*1),
    SVG(os.path.join(prefix, "noCFS_10^3_new_deltaVt", name)).scale(pt_to_mm).move(3.2*25.4, 2.4*25.4*1),
    SVG(os.path.join(prefix, "noCFS_10^1_new_deltaVt", name)).scale(pt_to_mm).move(3.2*25.4*2, 2.4*25.4*1),

    # --- panel labels ---
    Text("A", 5, 15, size=12, weight="bold").scale(pt_to_mm).move(0, 0),
    Text("B", 5, 15, size=12, weight="bold").scale(pt_to_mm).move(3.2*25.4, 0),
    Text("C", 5, 15, size=12, weight="bold").scale(pt_to_mm).move(3.2*25.4*2, 0),
    Text("D", 5, 15, size=12, weight="bold").scale(pt_to_mm).move(0, 2.4*25.4*1),
    Text("E", 5, 15, size=12, weight="bold").scale(pt_to_mm).move(3.2*25.4, 2.4*25.4*1),
    Text("F", 5, 15, size=12, weight="bold").scale(pt_to_mm).move(3.2*25.4*2, 2.4*25.4*1),
).save(os.path.join(output_folder, "figS.13.svg"))

In [ ]:
prefix = "./result/basic"
fname = "CFS_10^1/lineplots"
fname_lambAdjusted = "CFS_10^1_lambdaAdjusted/lineplots"
output_folder = "../2_graph/result"
pt_to_mm = 25.4 / 72

Figure(
    "250mm", "125mm",
    SVG(os.path.join(prefix, fname, "Nitrite_lineplot_n1.svg")).scale(pt_to_mm).move(0, 0),
    SVG(os.path.join(prefix, fname, "Nitrite_lineplot_n2.svg")).scale(pt_to_mm).move(3.2*25.4, 0),
    SVG(os.path.join(prefix, fname, "Nitrite_lineplot_n3.svg")).scale(pt_to_mm).move(3.2*25.4*2, 0),
    SVG(os.path.join(prefix, fname_lambAdjusted, "Nitrite_lineplot_n1.svg")).scale(pt_to_mm).move(0, 2.4*25.4*1),
    SVG(os.path.join(prefix, fname_lambAdjusted, "Nitrite_lineplot_n2.svg")).scale(pt_to_mm).move(3.2*25.4, 2.4*25.4*1),
    SVG(os.path.join(prefix, fname_lambAdjusted, "Nitrite_lineplot_n3.svg")).scale(pt_to_mm).move(3.2*25.4*2, 2.4*25.4*1),

    # --- panel labels ---
    Text("A", 5, 15, size=12, weight="bold").scale(pt_to_mm).move(0, 0),
    Text("B", 5, 15, size=12, weight="bold").scale(pt_to_mm).move(3.2*25.4, 0),
    Text("C", 5, 15, size=12, weight="bold").scale(pt_to_mm).move(3.2*25.4*2, 0),
    Text("D", 5, 15, size=12, weight="bold").scale(pt_to_mm).move(0, 2.4*25.4*1),
    Text("E", 5, 15, size=12, weight="bold").scale(pt_to_mm).move(3.2*25.4, 2.4*25.4*1),
    Text("F", 5, 15, size=12, weight="bold").scale(pt_to_mm).move(3.2*25.4*2, 2.4*25.4*1),

    # --- lambda labels ---
    Text(f"lambda=10", 0.85*25.4/pt_to_mm, 0.55*25.4/pt_to_mm, size=7).scale(pt_to_mm).move(0, 0),
    Text(f"lambda=10", 0.85*25.4/pt_to_mm, 0.55*25.4/pt_to_mm, size=7).scale(pt_to_mm).move(3.2*25.4, 0),
    Text(f"lambda=10", 0.85*25.4/pt_to_mm, 0.55*25.4/pt_to_mm, size=7).scale(pt_to_mm).move(3.2*25.4*2, 0),
    Text(f'lambda={params_for_weibull_CFS["CFS_10^1_lambdaAdjusted"]["init_cell_num"][0]:.2f}', 0.85*25.4/pt_to_mm, 0.55*25.4/pt_to_mm, size=7).scale(pt_to_mm).move(0, 2.4*25.4*1),
    Text(f'lambda={params_for_weibull_CFS["CFS_10^1_lambdaAdjusted"]["init_cell_num"][1]:.2f}', 0.85*25.4/pt_to_mm, 0.55*25.4/pt_to_mm, size=7).scale(pt_to_mm).move(3.2*25.4, 2.4*25.4*1),
    Text(f'lambda={params_for_weibull_CFS["CFS_10^1_lambdaAdjusted"]["init_cell_num"][2]:.2f}', 0.85*25.4/pt_to_mm, 0.55*25.4/pt_to_mm, size=7).scale(pt_to_mm).move(3.2*25.4*2, 2.4*25.4*1)
).save(os.path.join(output_folder, "figS.9_A-F.svg"))